# GEMS P6.1 — Performance Recovery & Structural Fault Discovery | Colab CPU (high-RAM)

Standalone runner for `gems_bench/p6/boost_diagnostics.py` (P6.1). It **reuses** the frozen W2 model, its sampled rows, its held-out and full-region maps, the M9 baseline context, the B0/B1 maps and the P6 boost benchmark from `MyDrive/GEMS_2`. It does **not** rerun W0–W2, M9, G01/G02 research or W3, and it never uploads a submission.

**Order:** 0 config → 1 Drive + strict clone → 2 artifact attachment + preflight → 3 X1 export parity → 4 X2 candidate recall → 5 X3 segment recovery → 6 X4 G01/G02 audit (+ explicit backfill, S1) → 7 X5 robustness → 8 selection → 9 export (OFF) → 10 report + Drive sync.

Every stage writes `outputs/p6_fast_track/recovery/<stage>/{manifest,status,decision}.json`, CSV / Parquet tables and PNG figures with their plot data. A stage whose fingerprint and outputs are unchanged reloads instead of recomputing, so after a Colab disconnect re-run sections 0–2 and continue. Nothing in this notebook has been executed by the authoring session: every number comes from your Colab run. Docs: `docs/p6/P6_RECOVERY_HANDOFF.md`.

## 0 — Configuration (edit before running)

First run: `preflight`, `export_parity`, `candidate_recall`, `report` only. `CONFIG['ref']` must resolve to the pushed P6.1 commit (pin the SHA in `EXPECT['commit']`); a missing branch is an error, never a silent fallback to `main`.

In [ ]:
CONFIG = {
    'repo': 'mybenkhadda/GEMS---drivendata',
    'ref': 'p6-recovery',            # branch with the P6.1 code, or (better) its exact commit SHA
    'drive_root': '/content/drive/MyDrive/GEMS_2',
    'local_root': '/content/gems_work',
    'run_mode': 'PREPARE_DATA',
    'failure_policy': 'interactive',
    'download_missing': False,
    'run_advanced_models': False,
    'train_baselines': [],
    'run_full_inference': False,
    'run_submission': False,
    'feature_stage': 'lazy',
    'use_gpu': False,                # CPU runtime (high-RAM)
    'seed': 2026,
}
PRIVATE_REPO = False                 # True: Colab Secrets must contain GITHUB_TOKEN
DEM_CONFIG = {'enabled': False, 'mode': 'inventory_only', 'download': False}
M9_CONFIG = {'profile': 'PILOT', 'overwrite_existing': False}
P6_CONFIG = {'overwrite_existing': False}
BOOST_CONFIG = {'geology': {'backfill_missing': False}}
RECOVERY_CONFIG = {}                 # overrides of configs/p6/recovery.yaml (predeclared; avoid tuning here)

RECOVERY_RUN = {
    'preflight': True,
    'export_parity': True,
    'candidate_recall': True,
    'segment_recovery': False,
    'g02_repair_audit': False,
    'g02_backfill': False,
    'geology': False,
    'robustness': False,
    'selection': False,
    'export': False,
    'report': True,
}
RECOVERY_EXPORT = {'candidate_fp': None}   # paste selection/decision.json -> candidate_fp before export
EXPECT = {'notebook_version': 'p6r-nb.1', 'code_rev': 'p6b.1-r3', 'recovery_code_rev': 'p6r.1-r1',
          'commit': None}                  # set to the pushed SHA (strict)


## 1 — Google Drive, strict repository clone, version lock

In [ ]:
import base64, os, subprocess, sys
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
CODE = '/content/GEMS'
REPO_URL = f"https://github.com/{CONFIG['repo']}.git"

def _git_env():
    if not PRIVATE_REPO:
        return None
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN') or ''
    if not token:
        raise RuntimeError('PRIVATE_REPO=True but GITHUB_TOKEN is missing in Colab Secrets')
    hdr = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    return {**os.environ, 'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader',
            'GIT_CONFIG_VALUE_0': f'AUTHORIZATION: basic {hdr}'}

def git(*args, auth=False, check=True):
    r = subprocess.run(['git', *args], capture_output=True, text=True, env=_git_env() if auth else None)
    if check and r.returncode:
        raise RuntimeError(f"git {' '.join(args[:4])} failed:\n{r.stderr[-1500:]}")
    return r

if not Path(CODE, '.git').exists():
    git('clone', '-q', REPO_URL, CODE, auth=True)
git('-C', CODE, 'remote', 'set-url', 'origin', REPO_URL)
git('-C', CODE, 'fetch', '-q', '--tags', 'origin', auth=True)
ref = CONFIG['ref']
if git('-C', CODE, 'rev-parse', '--verify', '-q', f'refs/remotes/origin/{ref}', check=False).returncode == 0:
    git('-C', CODE, 'checkout', '-q', '-B', 'p6r_colab_active', f'origin/{ref}')
else:                                   # a SHA / tag must resolve; never fall back to main
    git('-C', CODE, 'checkout', '-q', '--detach', ref)
HEAD = git('-C', CODE, 'rev-parse', 'HEAD').stdout.strip()
if 'gems_bench' in sys.modules:
    raise RuntimeError('gems_bench already imported: restart the runtime (stale module code)')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{CODE}[colab]'], check=True)
if CODE not in sys.path:
    sys.path.insert(0, CODE)
os.environ['GEMS_CODE_COMMIT'] = HEAD
print('Requested:', ref, '| resolved SHA:', HEAD)


In [ ]:
import json
import pandas as pd
from IPython.display import Image, Markdown, display
from gems_bench.p6.boost_diagnostics import startup_check
START = startup_check(CONFIG['ref'], HEAD,
                      {**EXPECT, 'code_root': CODE,
                       'artifact_root': f"{CONFIG['drive_root']}/outputs/p6_fast_track/recovery"},
                      BOOST_CONFIG, RECOVERY_CONFIG)
print(json.dumps(START, indent=2, default=str))


## 2 — Artifact attachment (no retraining) + preflight

Reuses the M9 baseline context, B0/B1 maps, the frozen W2 candidate / rows / maps (`stack/`), `stack/maps/full_score.tif` and the boost benchmark (`boost/benchmark/`). Missing W2 inputs stop here: do **not** regenerate W2 in this notebook.

In [ ]:
from gems_bench.pipeline import Pipeline
from gems_bench.p6 import P6FastTrack

pipe = Pipeline.from_notebook(CONFIG, DEM_CONFIG)
pipe.step('bootstrap')
for step in ('raw_data', 'native_grid', 'spatial_folds', 'discovery_splits', 'feature_store'):
    pipe.step(step)                     # validate + stage cached Drive data, never rebuild
m9 = pipe.m9(M9_CONFIG)
m9.load()
p6 = P6FastTrack(m9, P6_CONFIG)
rc = p6.recovery(RECOVERY_CONFIG, BOOST_CONFIG)
bst = rc.bst
need = bst.stage()                      # stack/, boost/, submissions/, manifests/ from Drive
p6.m9.stage_in([f"{p6.cfg['output_dir']}/{rc.cfg['subdir']}"])   # earlier recovery outputs (resume)
missing = [k for k in ('stack/candidate.json', 'stack/features/rows.parquet') if not need.get(k)]
if need.get('stack/maps/*.npy', 0) != 6:
    missing.append(f"stack/maps/*.npy expected 6, got {need.get('stack/maps/*.npy', 0)}")
if missing:
    raise RuntimeError('Frozen W2 artifacts missing: ' + ', '.join(missing) + ' (restore them on Drive; W2 is not rerun)')
print('Recovery root:', rc.root)
print('Champion:', json.dumps(bst.champion(), indent=2, default=str))
display(p6.budget.table())


In [ ]:
if RECOVERY_RUN['preflight']:
    pre = rc.preflight()
    print(json.dumps(pre['ready'], indent=2))
    print('Failed checks:', pre['failed_checks'])
    display(rc.load_table('dependency_check', 'preflight'))
    if not pre['ready']['candidate_recall']:
        print('STOP: core inputs missing (boost stage A must be complete; run the boost notebook A cell first).')


## 3 — X1: deployable W2 policies and export parity

Policies A–E (+ fallback reference) re-evaluated from the saved boost grid: for each outer zone ONE cell is chosen on non-overlapping zones under the predeclared objective (`disc_geo_floor`) and applied to the same zone in **both** tasks. `separate` rows reproduce the boost numbers (disc and geo selected independently; flagged not deployable when the cells differ). `META` is the honest estimate of choosing the policy itself. Parity compares the training construction with the full-region (inference) construction.

In [ ]:
if RECOVERY_RUN['export_parity']:
    x1 = rc.export_parity()
    d = x1['decision']
    cols = ['policy', 'objective', 'role', 'deployable', 'disc_dti', 'disc_worst', 'disc_worst_zone', 'geo_dti',
            'systems_gained', 'systems_lost', 'reproduces_benchmark', 'deploy_cell']
    C = x1['policy_comparison']
    display(C[[c for c in cols if c in C]])
    print('Chosen:', d['chosen_policy'], '| objective:', d['objective'], '| verdict:', d['verdict'])
    print('META (choose policy, nested):', json.dumps(d['meta_nested'], default=str))
    print('Plan audit:', d['plan_status'], '| parity BLOCKED rows:', d['parity_blocked'])
    P = x1['train_inference_parity']
    display(P.groupby(['check', 'status']).size().rename('rows').reset_index())
    display(Image(filename=str(rc.root / 'export_parity' / 'policy_frontier.png')))


## 4 — X2: candidate-pool recall ceiling

Reachable systems per source and budget, the inventory union, the **oracle** (held-out labels pick pool segments: a diagnostic upper bound only, never used for selection) and the failure taxonomy with its threshold sensitivity. Reference annotations in the figures are evaluation-only.

In [ ]:
if RECOVERY_RUN['candidate_recall']:
    x2 = rc.candidate_recall()
    d = x2['decision']
    print('Verdict:', d.get('verdict'), '| dominant:', d.get('dominant_group'), '| next:', d.get('next_direction'))
    print('Shares by length:', json.dumps(d.get('share_by_length'), default=str))
    display(pd.DataFrame(d.get('sensitivity') or []))
    cb = x2['coverage_by_budget']
    display(cb[[c for c in ('source', 'rep', 'frac', 'dti', 'system_recall', 'length_weighted_recall',
                           'completely_missed') if c in cb]])
    display(x2['oracle_upper_bound'])
    display(x2['missed_system_taxonomy'].groupby('category').size().rename('systems').reset_index())
    display(Image(filename=str(rc.root / 'candidate_recall' / 'coverage_curves.png')))
    display(Image(filename=str(rc.root / 'candidate_recall' / 'missed_system_examples.png')))


## 5 — X3: segment-level recovery (explicit)

H0 pure W2, H1 confident fallback-only additions, H2 graph-supported additions — all at the area of the H0 cell. A strategy that only adds systems without a nested DTI gain (or with a worst-zone / FP regression) is REJECTED.

In [ ]:
if RECOVERY_RUN['segment_recovery']:
    x3 = rc.segment_recovery()
    H = x3['hybrid_comparison']
    display(H[[c for c in ('candidate', 'decision', 'disc_dti', 'disc_worst', 'geo_dti', 'systems_gained',
                           'systems_lost', 'fp_increase', 'length_km', 'reasons') if c in H]])
    print('Budget violations:', x3['decision']['budget_violations'], '| chosen:', x3['decision']['chosen'])
    display(Image(filename=str(rc.root / 'segments' / 'recovery_examples.png')))


## 6 — X4: G01 / G02 artifact audit, explicit zone-2 backfill, S1

The audit classifies each blocker (missing attachment, missing saved artifact, registry mismatch, source incompatibility, true dependency). `g01:edge` / `g01:surface` are virtual channels computed from raw bands: expected class `REGISTRY_MISMATCH`, not a missing input. The backfill runs only when the audit says `READY_FOR_BACKFILL` **and** `RECOVERY_RUN['g02_backfill']` is True; S1 (W2 + G02-C) runs first, S2/S3 only after S1 ACCEPT or a declared diagnostic reason.

In [ ]:
if RECOVERY_RUN['g02_repair_audit'] or RECOVERY_RUN['g02_backfill'] or RECOVERY_RUN['geology']:
    g = rc.geology_audit()
    print('Repair status:', g['repair_status'])
    print(json.dumps(g['blockers'], indent=2, default=str))
    display(rc.load_table('compatibility_report', 'geology'))
    if RECOVERY_RUN['g02_backfill']:
        done = rc.g02_backfill()           # raises unless READY_FOR_BACKFILL
        print('Backfilled:', done)
        g = rc.geology_audit()
        print('Repair status after backfill:', g['repair_status'])
    if RECOVERY_RUN['geology']:
        s1 = rc.geology_s1()
        print('S1 decision:', s1['decision'], s1['reasons'], '| variants run:', s1['variants_run'])
        display(rc.load_table('s1_ablation', 'geology'))


## 7 — X5: weak-zone robustness (gated)

The weakest zone is identified per evaluated policy (never assumed). Boost C1/C2 run only with `RECOVERY_CONFIG = {'robustness': {'run_boost_c': True}}`; no zone-specific tuning.

In [ ]:
if RECOVERY_RUN['robustness']:
    x5 = rc.robustness()
    display(pd.DataFrame(x5['weak_zone_by_policy']))
    print('Boost C:', x5['boost_c_run'], x5['boost_c_chosen'], x5['boost_c_accepted'])


## 8 — Selection

Gate: discovery DTI > max(W2 reference, matched W2 benchmark), worst zone ≥ 0.153414, geographic DTI ≥ 0.1758, one deployable policy, parity without BLOCKED rows (numeric parity run), planned export recipe. A leaderboard slot additionally needs +0.010. Oracle, historical fixed cells and split disc/geo policies are never promotable.

In [ ]:
if RECOVERY_RUN['selection']:
    sel = rc.select()
    print(json.dumps({k: sel[k] for k in ('chosen', 'replace_champion', 'slot_worthy', 'candidate_fp',
                                          'matched_w2_benchmark')}, indent=2, default=str))
    S = rc.load_table('candidates', 'selection')
    display(S[[c for c in ('name', 'role', 'disc_dti', 'disc_worst', 'geo_dti', 'gate', 'gate_reasons',
                           'slot_worthy', 'delta_vs_w2_recorded', 'delta_vs_best_matched_w2_benchmark') if c in S]])
else:
    assert not RECOVERY_RUN['export'], 'run selection before export'


## 9 — Submission export (OFF by default)

Requires selection PASS, `RECOVERY_EXPORT['candidate_fp']` equal to `selection/decision.json → candidate_fp`, a PASS plan audit and a PASS post-write audit (CRS, transform, shape, nodata, eligibility, area, sidecar = recipe, sha256). The historical W2 file is never touched; files are never overwritten. **Upload by hand** and type the public score into the leaderboard log.

In [ ]:
if RECOVERY_RUN['export']:
    out = rc.export(RECOVERY_EXPORT['candidate_fp'])
    print(json.dumps(out, indent=2, default=str))
    print(json.dumps(rc.read_json('export_parity', 'export_audit'), indent=2, default=str)[:4000])


## 10 — Report, registry, Drive synchronisation

In [ ]:
if RECOVERY_RUN['report']:
    rep = rc.report()
    display(rep['status'])
    display(Markdown(rep['markdown']))
p6.push()                                # recovery/ (and submissions/ after an export) to Drive
display(p6.budget.table())
print('Artifacts:', rc.root)


## Execution checklist

1. First run: `preflight`, `export_parity`, `candidate_recall`, `report`. Read `export_parity/decision.json` and `candidate_recall/decision.json`.
2. `segment_recovery=True` once X1 has a chosen policy (X3 uses its objective).
3. `g02_repair_audit=True`; if `READY_FOR_BACKFILL`, `g02_backfill=True` then `geology=True` (S1 only).
4. `robustness=True` (and `run_boost_c` only when justified).
5. `selection=True` with export OFF; if PASS and slot-worthy, paste `candidate_fp`, set `export=True`, rerun 8–9. Upload manually.

Restart after a disconnect: run 0 → 2, then any section; finished stages reload (fingerprint + outputs checked). Reference numbers quoted in `configs/p6/recovery.yaml` are recorded offline values, not results of this notebook.